In [37]:
import requests
from config.settings import settings
import pandas as pd
import numpy as np
import json
from config.leagues import LEAGUES
import xml.etree.ElementTree as ET
import feedparser

### Getting the top five leagues

- Find out the structure of the leagues and UCL data from the API
- Map out what identifiers exist in the data
- Write to a csv so it's more readable 
- Create dictionary as source of truth

In [3]:
COMP_LEVEL_API_URL = "api/v2/leagues/"
TOP_FIVE_LEAGUE_NATIONS = ["England", "Spain", "Germany", "Italy", "France"]

summary_df = []
for nation in TOP_FIVE_LEAGUE_NATIONS:
    r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country={nation}", headers={
        'Authorization': f"Token {settings.bzzorio_api_key}",
        })
    print(f"Calling {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country={nation}")
    print(f"API Call returned Status Code: {r.status_code}")
    if r.status_code == 200:
        data = json.loads(json.dumps(r.json().get("results", [])))
        df = pd.json_normalize(data)
        summary_df.append(df)


Calling https://sports.bzzoiro.com/api/v2/leagues/?country=England
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Spain
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Germany
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Italy
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=France
API Call returned Status Code: 200


In [4]:
new_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country=Spain", headers={
    'Authorization': f"Token {settings.bzzorio_api_key}",
})
print(f"Fetching Spanish Comps At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country=Spain")
print(f"API Call returned Status Code: {new_r.status_code}")
if new_r.status_code == 200:
        data = json.loads(json.dumps(new_r.json().get("results", [])))
        df = pd.json_normalize(data)
        summary_df.append(df)
summary_df = pd.concat(summary_df, ignore_index=True)

Fetching Spanish Comps At: https://sports.bzzoiro.com/api/v2/leagues/?country=Spain
API Call returned Status Code: 200


In [5]:
summary_df = summary_df[["id", "name", "country", "current_season.id"]]
summary_df.sort_values(by=['id'], inplace=True)
summary_df.to_csv("competitions.csv", index=False)

### Getting the league tables

- How to get individual league tables and standings
- Mapping them to league ids

In [6]:
la_liga_id = LEAGUES.get("la_liga", {})["bzzorio_id"]
new_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/season", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
})
standings_r = None
print(f"Fetching La Liga Season Data At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/season")
print(f"API Call returned Status Code: {new_r.status_code}")

Fetching La Liga Season Data At: https://sports.bzzoiro.com/api/v2/leagues/3/season
API Call returned Status Code: 200


In [7]:
if new_r.status_code == 200:
    season_id = new_r.json().get("season", {}).get("id")
    standings_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/standings/?season={season_id}", headers={
        "Authorization": f"Token {settings.bzzorio_api_key}"  
        })
    print(f"Fetching La Liga Standings Data At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/standings/?season={season_id}")
    print(f"API Call returned Status Code: {standings_r.status_code}")

Fetching La Liga Standings Data At: https://sports.bzzoiro.com/api/v2/leagues/3/standings/?season=1307
API Call returned Status Code: 200


In [8]:
if standings_r and standings_r.status_code == 200:
    standings_data = json.loads(json.dumps(standings_r.json().get("standings", [])))
    standings_df = pd.json_normalize(standings_data)
    print(standings_df.head(20))
    standings_df.to_csv("epl_standings.csv", index=False)

    position  team_id              team_name  played  won  drawn  lost  gf  \
0          1       44           FC Barcelona       3    3      0     0  12   
1          2       57            Real Madrid       3    3      0     0  10   
2          3       54        Atlético Madrid       3    2      1     0   7   
3          4       45       Deportivo Alavés       3    2      1     0   5   
4          5       58                Osasuna       3    2      1     0   3   
5          6       52                Sevilla       3    2      0     1   6   
6          7       56             Real Betis       3    2      0     1   4   
7          8     1260  Deportivo de A Coruña       3    1      2     0   5   
8          9       46             Levante UD       3    1      1     1   5   
9         10     1400       Real Racing Club       3    1      1     1   5   
10        11       53               Espanyol       3    1      0     2   5   
11        12       51          Athletic Club       3    1      0

### Getting fixture lists for the next two weeks

- How to get fixtures
- What data does the matches endpoint have
- Mapping match ids to get matches as needed

In [9]:
EVENT_LEVEL_API_URL = "api/v2/events/"
today = pd.Timestamp.now().strftime("%Y-%m-%d")
two_weeks_from_now = (pd.Timestamp.now() + pd.Timedelta(days=14)).strftime("%Y-%m-%d")
fixtures_r = requests.get(f"{settings.bzzorio_base_url}{EVENT_LEVEL_API_URL}?league_id={la_liga_id}&season={season_id}&date_from={today}&date_to={two_weeks_from_now}", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
    })
print(f"Fetching La Liga Fixtures Data At: {settings.bzzorio_base_url}{EVENT_LEVEL_API_URL}?league_id={la_liga_id}&season={season_id}&date_from={today}&date_to={two_weeks_from_now}")
print(f"API Call returned Status Code: {fixtures_r.status_code}")

Fetching La Liga Fixtures Data At: https://sports.bzzoiro.com/api/v2/events/?league_id=3&season=1307&date_from=2026-09-02&date_to=2026-09-16
API Call returned Status Code: 200


In [10]:
if fixtures_r and fixtures_r.status_code == 200:
    fixtures_data = fixtures_r.json().get("results", [])
    
    # Save the original nested data beautifully
    with open("la_liga_fixtures.json", "w") as f:
        json.dump(fixtures_data, f, indent=4)

### Getting stats for la liga

- Structure of stats data
- Mapping of stats data

In [16]:
GET_LA_LIGA_STATS = f"api/v2/leagues/{la_liga_id}/top/"

top_scorer = requests.get(f"{settings.bzzorio_base_url}{GET_LA_LIGA_STATS}scorers", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
    })
print(f"Fetching La Liga Top Scorers Data At: {settings.bzzorio_base_url}{GET_LA_LIGA_STATS}scorers")
print(f"API Call returned Status Code: {top_scorer.status_code}")
print()

if top_scorer and top_scorer.status_code == 200:
    top_scorer_df = pd.json_normalize(top_scorer.json().get("leaders", []))
    print(top_scorer_df.head(20))
    top_scorer_df.to_csv("la_liga_top_scorers.csv", index=False)
    

Fetching La Liga Top Scorers Data At: https://sports.bzzoiro.com/api/v2/leagues/3/top/scorers
API Call returned Status Code: 200

    rank  player_id                player_name position  team_id  \
0      1        747                   Raphinha        M       44   
1      2        594              Kylian Mbappé        F       57   
2      3        769                 Alex Baena        M       54   
3      4       2669               Ante Budimir        F       58   
4      5       3663               Fermín López        M       44   
5      6       1932  Pierre-Emerick Aubameyang        F     1260   
6      7       1284          Roberto Fernández        F       53   
7      8        918             Sergio Camello        F       40   
8      9       2332              Yassir Zabiri        F     1400   
9     10        592            Jude Bellingham        M       57   
10    11        745               Lamine Yamal        M       44   
11    12       2364               Mariano Díaz        

In [17]:
top_assists = requests.get(f"{settings.bzzorio_base_url}{GET_LA_LIGA_STATS}assists", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
    })
print(f"Fetching La Liga Top Assists Data At: {settings.bzzorio_base_url}{GET_LA_LIGA_STATS}assists")
print(f"API Call returned Status Code: {top_assists.status_code}")
print()

if top_assists and top_assists.status_code == 200:
    top_assists_df = pd.json_normalize(top_assists.json().get("leaders", []))
    print(top_assists_df.head(20))
    top_assists_df.to_csv("la_liga_top_assists.csv", index=False)

Fetching La Liga Top Assists Data At: https://sports.bzzoiro.com/api/v2/leagues/3/top/assists
API Call returned Status Code: 200

    rank  player_id         player_name position  team_id         team_name  \
0      1        711      Anthony Gordon        F       44      FC Barcelona   
1      2       1294    Javier Hernández        M       53          Espanyol   
2      3        592     Jude Bellingham        M       57       Real Madrid   
3      4       2364        Mariano Díaz        F       45  Deportivo Alavés   
4      5       3890     Mikel Oyarzabal        F       48     Real Sociedad   
5      6        595     Vinícius Júnior        F       57       Real Madrid   
6      7       3980         Xavi Espart        D       44      FC Barcelona   
7      8       1772     Alberto Moleiro        M       41        Villarreal   
8      9        769          Alex Baena        M       54   Atlético Madrid   
9     10        912       Álvaro García        M       40    Rayo Vallecano   
1

In [18]:
most_fouls = requests.get(f"{settings.bzzorio_base_url}{GET_LA_LIGA_STATS}fouls", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"    
})
print(f"Fetching La Liga Most Fouls Data At: {settings.bzzorio_base_url}{GET_LA_LIGA_STATS}fouls")
print(f"API Call returned Status Code: {most_fouls.status_code}")
print()
if most_fouls and most_fouls.status_code == 200:
    most_fouls_df = pd.json_normalize(most_fouls.json().get("leaders", []))
    print(most_fouls_df.head(20))
    most_fouls_df.to_csv("la_liga_most_fouls.csv", index=False)

Fetching La Liga Most Fouls Data At: https://sports.bzzoiro.com/api/v2/leagues/3/top/fouls
API Call returned Status Code: 200

    rank  player_id          player_name position  team_id  \
0      1       1742         Jon Aramburu        D       48   
1      2      16386       Gustavo Puerta        M     1400   
2      3       2348       Antonio Blanco        M       45   
3      4       3801        Gabriel Suazo        D       52   
4      5       3352       Gonzalo Villar        M       55   
5      6       1294     Javier Hernández        M       53   
6      7       1559        Lucien Agoumé        M       52   
7      8      32049  Beñat Gerenabarrena        M       51   
8      9      27288          Luismi Cruz        M     1260   
9     10       1275       Omar El Hilali        D       53   
10    11       1118         Orel Mangala        M       50   
11    12       2352  Abderrahman Rebbach        M       45   
12    13      30427       Álex Calatrava        M       53   
13   

### Getting news feeds from FOX, ESPN, BBC, SKY, CBS Sport

- Get news feeds from the above mentioned sites
- Map the structure for use 

In [ ]:
NEWS_FEEDS = {
    "BBC": "https://feeds.bbci.co.uk/sport/football/rss.xml",
    "SKY Sports": "https://www.skysports.com/rss/12040",
    "CBS Sports": "https://www.cbssports.com/rss/headlines/soccer/",
    "ESPN Sports": "https://www.espn.com/espn/rss/soccer/news",
    "FOX Sports": "https://api.foxsports.com/v2/content/optimized-rss?partnerKey=MB0Wehpmuj2lUhuRhQaafhBjAJqaPU244mlTDK1i&size=30&tags=fs%2Fsoccer"
    
}

NAMESPACES = {"media": "http://search.yahoo.com/mrss/"}

article_rss_url = {}

resp = requests.get("https://feeds.bbci.co.uk/sport/football/rss.xml")
root = ET.fromstring(resp.content)

feed = feedparser.parse("https://feeds.bbci.co.uk/sport/football/rss.xml")

for entry in feed.entries:
    article_rss_url[entry.title] = {"link": entry.link, "published": entry.published, "summary": entry.summary}

for item in root.findall(".//item"):
    title = item.find("title").text
    thumbnail = item.find("media:thumbnail", NAMESPACES)
    image_url = thumbnail.get("url") if thumbnail is not None else None
    if title in article_rss_url:
        article_rss_url[title]["image_url"] = image_url
    
for title, details in article_rss_url.items():
    print(f"Title: {title}")
    print(f"Link: {details['link']}")
    print(f"Published: {details['published']}")
    print(f"Summary: {details['summary']}")
    print(f"Image URL: {details.get('image_url')}")
    print()

Title: Man City buy costly new midfield - but have they overlooked Haaland cover?
Link: https://www.bbc.co.uk/sport/football/articles/cpq02z298p7o?at_medium=RSS&at_campaign=rss
Published: Wed, 02 Sep 2026 22:07:52 GMT
Summary: Manchester City have completed a squad overhaul including a fully revamped midfield to the tune of a Premier League record £458m, but that eyewatering number does not tell the full story.
Image URL: https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/dfdf/live/8babcca0-a6cb-11f1-b0c9-07afcdd8053c.png

Title: How Everton's deadline day debacle leaves owners facing fan mutiny
Link: https://www.bbc.co.uk/sport/football/articles/c9861n79kepo?at_medium=RSS&at_campaign=rss
Published: Wed, 02 Sep 2026 12:33:25 GMT
Summary: Everton's transfer deadline day debacle over Folarin Balogun leaves owners The Friedkin facing fan mutiny, says chief football writer Phil McNulty.
Image URL: https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/6163/live/609a7b90-a6be-11f1-ae1e-219da1